# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [2,4,6] -> [1, 2, 3] [0, 1, 1] -> операция происходит с каждым элементом
2. [[1,1,1,1],[1,1,1,1],[1,1,1,1]] -> [4, 4, 4] -> asix=1 значит схлопываем по столбцам
3. [0,1,2]+[[0],[1],[2]] -> broadcast -> [[0,1,2],[1,2,3],[2,3,4]] -> вектор бродкастится на матрицу
4. [1,4,9] -> ([1,2,3]) -> 6 -> srt() это квадратный корень из числа, операция происходит с каждым элементом
5. [1,2,3,4] -> [1,4,9,16] -> (30/4) -> 7.5 -> каждый элемент возводится в квадрат -> среднее арифметическое

In [30]:
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [31]:
np.arange(3).reshape(3, 1)

array([[0],
       [1],
       [2]])

In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [4]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [5]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [19]:
# 1
print(clean.sum())
print((amount[(clean) & (month == '2026-09')].sum()-amount[(clean) & (month == '2026-10')].sum())/amount[(clean) & (month == '2026-09')].sum()*100)

4739
2.337713632137185


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [7]:
# 2

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [8]:
# 3

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [9]:
# 4

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [10]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [11]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [12]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [13]:
# 8